# IBM HR Attrition Analysis

Primary dataset: IBM HR Analytics Employee Attrition benchmark. This notebook uses the 1,470-row, 35-column CSV in `data/raw/ibm_hr_attrition.csv`.

The repository also retains a separate 50-row exercise. Its notebook and validator are not used here.

## Objective and interpretation limits

Explore attrition labels across departments, roles, overtime, satisfaction codes and sample income quartiles. The analysis is descriptive. It does not establish why people leave or estimate a current employer's attrition rate.

The source documentation does not define every ordinal scale or monetary unit. Do not infer units or scale labels. Do not use these benchmark associations to make decisions about real employees. See `docs/DATASET_PROVENANCE.md`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.ibm_hr_data import (
    load_ibm_hr_data,
    summarize_attrition,
    grouped_attrition,
    income_quartile_attrition,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
DATA_PATH = Path("data/raw/ibm_hr_attrition.csv")
OUTPUT_PATH = Path("outputs")
OUTPUT_PATH.mkdir(exist_ok=True)
print("Libraries imported.")

## 1. Load and validate the benchmark

In [ ]:
df = load_ibm_hr_data(DATA_PATH)
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print("Missing cells:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Unique EmployeeNumber values:", int(df["EmployeeNumber"].nunique()))
display(df.head())
assert df.shape == (1470, 35)
assert df["EmployeeNumber"].is_unique
assert not df.isna().any().any()

## 2. Dataset structure and numeric profile

The identifier and source fields that are constant in this dataset are excluded from numeric feature summaries.

In [ ]:
summary = summarize_attrition(df)
print("Numeric measures used:")
print(summary["numeric_measure_columns"])
print("Excluded identifier:", summary["identifier_excluded_from_numeric_summary"])
print("Excluded constant fields:", summary["constant_fields_excluded_from_numeric_summary"])
display(summary["numeric_summary"].T)

## 3. Overall attrition label distribution

In [ ]:
attrition_counts = df["Attrition"].value_counts().rename_axis("Attrition").reset_index(name="Records")
display(attrition_counts)
print("Attrition records:", summary["attrition_records"])
print("Non-attrition records:", summary["non_attrition_records"])
print("Attrition share in this benchmark (%):", summary["attrition_share_percent"])
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=attrition_counts, x="Attrition", y="Records", hue="Attrition", legend=False, ax=ax)
ax.set(title="Attrition label count in the benchmark", xlabel="Attrition label", ylabel="Record count")
fig.tight_layout()
fig.savefig(OUTPUT_PATH / "notebook_attrition_overall.png", dpi=150)
plt.show()

## 4. Attrition by department and job role

Percentages are within each category. Read each percentage together with its record count.

In [ ]:
by_department = grouped_attrition(df, "Department")
by_job_role = grouped_attrition(df, "JobRole")
display(by_department)
display(by_job_role)
fig, ax = plt.subplots(figsize=(8, 5))
plot_data = by_department.sort_values("attrition_share_percent")
ax.barh(plot_data["Department"], plot_data["attrition_share_percent"])
ax.set(title="Attrition share by department", xlabel="Attrition share (%)", ylabel="Department")
fig.tight_layout()
fig.savefig(OUTPUT_PATH / "notebook_attrition_by_department.png", dpi=150)
plt.show()

## 5. Attrition by overtime and satisfaction code

Satisfaction codes are retained as numeric categories because the source material does not document their full labels.

In [ ]:
by_overtime = grouped_attrition(df, "OverTime")
by_satisfaction = grouped_attrition(df, "JobSatisfaction")
display(by_overtime)
display(by_satisfaction)

## 6. Attrition across income quartiles

Quartiles are derived from this sample's `MonthlyIncome` distribution. The source does not document the currency, so the table reports the original values without a currency symbol.

In [ ]:
by_income = income_quartile_attrition(df)
display(by_income)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(by_income["income_quartile"].astype(str), by_income["attrition_share_percent"])
ax.set(title="Attrition share by sample income quartile", xlabel="Sample-defined income quartile", ylabel="Attrition share (%)")
fig.tight_layout()
fig.savefig(OUTPUT_PATH / "notebook_attrition_by_income_quartile.png", dpi=150)
plt.show()

## 7. Business questions and concise findings

In [ ]:
print(f"1. The sample contains {summary['records']:,} records and {summary['fields']} fields.")
print(f"2. {summary['attrition_records']:,} records are labelled Attrition=Yes ({summary['attrition_share_percent']:.2f}% of this sample).")
print("3. Department comparisons:")
display(by_department[["Department", "records", "attrition_records", "attrition_share_percent"]])
print("4. Overtime comparison:")
display(by_overtime[["OverTime", "records", "attrition_records", "attrition_share_percent"]])
print("5. Income quartile comparison:")
display(by_income)
print("These are descriptive benchmark patterns. They do not demonstrate causation.")

## 8. Export reusable tables and JSON summary

In [ ]:
tables = {
    "by_department": by_department,
    "by_job_role": by_job_role,
    "by_overtime": by_overtime,
    "by_job_satisfaction_code": by_satisfaction,
    "by_income_quartile": by_income,
}
for name, table in tables.items():
    table.to_csv(OUTPUT_PATH / f"notebook_{name}.csv", index=False)
print("Notebook tables and charts saved to:", OUTPUT_PATH.resolve())
print(summary["interpretation_note"])

## 9. Conclusion and limitations

This notebook answers descriptive questions about the supplied IBM HR Attrition benchmark. It validates the dataset, profiles numeric fields, compares attrition labels across business categories and exports tables and charts.

The data is static and its upstream documentation does not define every scale label, monetary unit or measurement period. It is not a current employer workforce sample. The results do not identify causes, do not prove that one group is at greater risk in a real organization, and must not be used to make employment decisions.

No prediction model is built in this notebook. A later model would need a documented objective, leakage checks, a suitable validation strategy, baseline comparisons, subgroup error analysis and clear limitations.